<a href="https://colab.research.google.com/github/HopeSilkina/deposits_forecast_project/blob/main/notebooks/01_EDA_Modeling_Deposits_Forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FORECASTING HOUSEHOLD DEPOSITS IN RUSSIA
# Data Scientist Portfolio Project
# Author: Nadezhda Silkina
# Date: 2026
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")

# ============================================================
# 2. LOAD DATA
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')

# Convert date column
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# Sort by date
df.sort_index(inplace=True)

print(f"✅ Data loaded. Records: {len(df)}")
print(f"Period: {df.index.min()} to {df.index.max()}")
print("\nFirst 5 rows:")
print(df.head())

# ============================================================
# 3. EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

print("\n" + "="*60)
print("3. EXPLORATORY DATA ANALYSIS")
print("="*60)

# 3.1. Summary statistics
print("\n📊 Summary Statistics:")
print(df.describe())

# 3.2. Check for missing values
print("\n📊 Missing Values:")
print(df.isnull().sum())

# 3.3. Correlation matrix (heatmap)
plt.figure(figsize=(12, 10))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of All Features', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# 3.4. Time series visualization
fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(14, 12))
axes = axes.flatten()

# Select only original features for time series
original_features = ['DEPOS', 'WAGE', 'SERV', 'DEP1', 'CRED1', 'CPI', 'USDind', 'UNEM', 'IPI', 'IMP']

for idx, col in enumerate(original_features):
    ax = axes[idx]
    ax.plot(df.index, df[col], linewidth=1.5)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
    ax.grid(True, alpha=0.3)

# Remove empty subplots (if any)
for idx in range(len(original_features), len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Time Series of All Indicators', fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig('time_series_all.png', dpi=300, bbox_inches='tight')
plt.show()

# 3.5. Scatter plots: DEPOS vs each original feature
print("\n📊 Generating scatter plots: DEPOS vs each predictor...")

# Original features (excluding DEPOS)
predictors = ['WAGE', 'SERV', 'DEP1', 'CRED1', 'CPI', 'USDind', 'UNEM', 'IPI', 'IMP']
n_features = len(predictors)

# Calculate grid size (3x3 = 9)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(predictors):
    ax = axes[idx]
    ax.scatter(df[col], df['DEPOS'], alpha=0.6, s=30, color='steelblue', edgecolor='white')

    # Add trend line (linear regression)
    mask = ~(df[col].isna() | df['DEPOS'].isna())
    if mask.sum() > 1:
        z = np.polyfit(df.loc[mask, col], df.loc[mask, 'DEPOS'], 1)
        p = np.poly1d(z)
        x_sorted = np.sort(df.loc[mask, col])
        ax.plot(x_sorted, p(x_sorted), "r--", linewidth=1.5,
                label=f'R² = {np.corrcoef(df.loc[mask, col], df.loc[mask, "DEPOS"])[0,1]**2:.3f}')

    ax.set_xlabel(col, fontsize=10)
    ax.set_ylabel('DEPOS', fontsize=10)
    ax.set_title(f'DEPOS vs {col}', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Remove empty subplots (if any)
for idx in range(n_features, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Scatter Plots: Household Deposits vs Macroeconomic Indicators',
             fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Scatter plots saved to 'scatter_plots.png'")

# 3.6. Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['DEPOS'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_title('Histogram: Household Deposits (DEPOS)')
axes[0].set_xlabel('Billion RUB')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df['DEPOS'])
axes[1].set_title('Boxplot: Household Deposits (DEPOS)')
axes[1].set_ylabel('Billion RUB')

plt.tight_layout()
plt.savefig('depos_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ EDA completed")

# ============================================================
# 4. ECONOMETRIC DIAGNOSTICS
# ============================================================

print("\n" + "="*60)
print("4. ECONOMETRIC DIAGNOSTICS")
print("="*60)

# 4.1. Stationarity test (Augmented Dickey-Fuller)
print("\n🔍 Augmented Dickey-Fuller Test for DEPOS:")

def adf_test(series, series_name):
    result = adfuller(series, autolag='AIC')
    print(f"\n  {series_name}:")
    print(f"    ADF Statistic: {result[0]:.4f}")
    print(f"    p-value: {result[1]:.4f}")
    print(f"    Critical Values:")
    for key, value in result[4].items():
        print(f"      {key}: {value:.4f}")
    print(f"    Conclusion: {'Stationary ✅' if result[1] < 0.05 else 'Non-stationary ❌'}")

adf_test(df['DEPOS'], 'DEPOS')

# 4.2. Multicollinearity (VIF)
print("\n🔍 Multicollinearity (VIF):")

X_vif = df.drop('DEPOS', axis=1)
# Add constant for VIF
X_vif_with_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data['feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print(vif_data.sort_values('VIF', ascending=False))
print("\n📌 Interpretation: VIF > 10 indicates strong multicollinearity")

# ============================================================
# 5. DATA PREPARATION FOR MODELING
# ============================================================

print("\n" + "="*60)
print("5. DATA PREPARATION FOR MODELING")
print("="*60)

# 5.1. Log transformation of target variable
df['DEPOS_log'] = np.log(df['DEPOS'])

# 5.2. Add lag features
for lag in [1, 3, 6, 12]:
    df[f'DEPOS_lag_{lag}'] = df['DEPOS'].shift(lag)

# 5.3. Prepare features and target
X = df.drop(['DEPOS', 'DEPOS_log'], axis=1).dropna()
y = df.loc[X.index, 'DEPOS']

print(f"📊 X shape: {X.shape}")
print(f"📊 y shape: {y.shape}")

# 5.4. Train-test split (last 12 months for testing)
train_size = len(X) - 12
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"\n📊 Training set: {len(X_train)} records")
print(f"📊 Test set: {len(X_test)} records")
print(f"📊 Test period: {X.index[train_size]} — {X.index[-1]}")

# 5.5. Feature scaling (for Ridge regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ============================================================
# 6. MODEL TRAINING AND COMPARISON
# ============================================================

print("\n" + "="*60)
print("6. MODEL TRAINING AND COMPARISON")
print("="*60)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\n🔧 Training {name}...")

    # Use scaled data for Ridge, original for others
    if name == 'Ridge Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    # Evaluation metrics
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    results[name] = {
        'R²': r2,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': f"{mape:.2f}%",
        'predictions': y_pred,
        'model': model
    }

    print(f"  ✅ R²: {r2:.4f}")
    print(f"  ✅ MAE: {mae:.2f} billion RUB")
    print(f"  ✅ RMSE: {rmse:.2f} billion RUB")
    print(f"  ✅ MAPE: {mape:.2f}%")

# 6.1. Comparison table
results_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k not in ['predictions', 'model']}
    for name, res in results.items()
}).T

print("\n" + "="*60)
print("📊 MODEL COMPARISON TABLE:")
print("="*60)
print(results_df.round(4))

# 6.2. Durbin-Watson test (for best model)
best_model_name = results_df['R²'].idxmax()
best_residuals = y_test - results[best_model_name]['predictions']
dw_stat = durbin_watson(best_residuals)

print(f"\n🔍 Durbin-Watson test for best model ({best_model_name}):")
print(f"  DW = {dw_stat:.3f} (ideal value = 2.0)")
print(f"  Conclusion: {'No autocorrelation ✅' if 1.5 < dw_stat < 2.5 else 'Autocorrelation detected ⚠️'}")

# ============================================================
# 6.3. RIDGE REGRESSION COEFFICIENTS (Interpretability)
# ============================================================

print("\n" + "="*60)
print("6.3. RIDGE REGRESSION COEFFICIENTS")
print("="*60)

ridge_model = results['Ridge Regression']['model']
feature_names = X.columns

# Get coefficients (on scaled features)
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': ridge_model.coef_
}).sort_values('coefficient', ascending=False)

print("\n📊 Ridge Regression Coefficients (scaled features):")
print(coef_df.to_string(index=False))

print("\n📌 Interpretation:")
print("   Positive coefficient → DEPOS INCREASES when this factor increases")
print("   Negative coefficient → DEPOS DECREASES when this factor increases")

print("\n🔺 TOP 5 DRIVERS (increase deposits):")
print(coef_df.head(5).to_string(index=False))

print("\n🔻 TOP 5 DRAGGERS (decrease deposits):")
print(coef_df.tail(5).to_string(index=False))

# ============================================================
# 6.4. OPTIMAL ALPHA FOR RIDGE REGRESSION (Hyperparameter Tuning)
# ============================================================

print("\n" + "="*60)
print("6.4. OPTIMAL ALPHA FOR RIDGE REGRESSION")
print("="*60)

from sklearn.linear_model import RidgeCV

# Try a range of alpha values
alphas = np.logspace(-3, 3, 50)  # from 0.001 to 1000

# RidgeCV with cross-validation (5 folds)
ridge_cv = RidgeCV(alphas=alphas, scoring='neg_mean_squared_error', cv=5)
ridge_cv.fit(X_train_scaled, y_train)

best_alpha = ridge_cv.alpha_
print(f"\n✅ Best alpha from cross-validation: {best_alpha:.4f}")

# Train model with best alpha
ridge_optimal = Ridge(alpha=best_alpha)
ridge_optimal.fit(X_train_scaled, y_train)
y_pred_optimal = ridge_optimal.predict(X_test_scaled)

# Compare with original Ridge (alpha=1.0)
r2_original = results['Ridge Regression']['R²']
r2_optimal = r2_score(y_test, y_pred_optimal)
rmse_original = results['Ridge Regression']['RMSE']
rmse_optimal = np.sqrt(mean_squared_error(y_test, y_pred_optimal))

print(f"\n📊 Comparison:")
print(f"   Original Ridge (alpha=1.0):  R² = {r2_original:.4f}, RMSE = {rmse_original:.2f}")
print(f"   Optimal Ridge (alpha={best_alpha:.4f}): R² = {r2_optimal:.4f}, RMSE = {rmse_optimal:.2f}")

# If optimal is better, update the best model
if r2_optimal > r2_original:
    print("\n✅ Optimal Ridge outperforms original! Updating best model...")
    results['Ridge Regression (Optimal)'] = {
        'R²': r2_optimal,
        'MAE': mean_absolute_error(y_test, y_pred_optimal),
        'RMSE': rmse_optimal,
        'MAPE': f"{np.mean(np.abs((y_test - y_pred_optimal) / y_test)) * 100:.2f}%",
        'predictions': y_pred_optimal,
        'model': ridge_optimal
    }
    best_model_name = 'Ridge Regression (Optimal)'
    best_pred = y_pred_optimal
else:
    print("\nℹ️ Original Ridge (alpha=1.0) is already optimal.")

# ============================================================
# 6.5. FEATURE SELECTION FOR LINEAR REGRESSION (p-values)
# ============================================================

print("\n" + "="*60)
print("6.5. FEATURE SELECTION FOR LINEAR REGRESSION")
print("="*60)

# Add constant for statsmodels
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

# Fit OLS with statsmodels
ols_full = sm.OLS(y_train, X_train_const).fit()

print("\n📊 Full Linear Regression Summary (p-values):")
print("="*60)
print(ols_full.summary())

# Extract p-values
p_values = ols_full.pvalues
significant_features = p_values[p_values < 0.05].index.tolist()

# Remove 'const' from list
significant_features = [f for f in significant_features if f != 'const']

print(f"\n📊 Features with p-value < 0.05 (statistically significant):")
print(f"   {len(significant_features)} features: {significant_features}")

# Build reduced model with only significant features
if len(significant_features) > 0:
    X_train_reduced = X_train[significant_features]
    X_test_reduced = X_test[significant_features]

    # Train reduced linear regression
    lr_reduced = LinearRegression()
    lr_reduced.fit(X_train_reduced, y_train)
    y_pred_reduced = lr_reduced.predict(X_test_reduced)

    # Metrics
    r2_reduced = r2_score(y_test, y_pred_reduced)
    rmse_reduced = np.sqrt(mean_squared_error(y_test, y_pred_reduced))
    mae_reduced = mean_absolute_error(y_test, y_pred_reduced)

    print(f"\n📊 Reduced Linear Regression (only significant features):")
    print(f"   R² = {r2_reduced:.4f}")
    print(f"   RMSE = {rmse_reduced:.2f} billion RUB")
    print(f"   MAE = {mae_reduced:.2f} billion RUB")

    # Compare with original linear regression
    r2_original_lr = results['Linear Regression']['R²']
    rmse_original_lr = results['Linear Regression']['RMSE']

    print(f"\n📊 Comparison with Full Linear Regression:")
    print(f"   Full model:   R² = {r2_original_lr:.4f}, RMSE = {rmse_original_lr:.2f}")
    print(f"   Reduced model: R² = {r2_reduced:.4f}, RMSE = {rmse_reduced:.2f}")

    if r2_reduced > r2_original_lr:
        print("   ✅ Reduced model is better (removed noise)")
    else:
        print("   ℹ️ Full model is better (all features contribute)")

    # Show coefficients for reduced model
    coef_reduced = pd.DataFrame({
        'feature': significant_features,
        'coefficient': lr_reduced.coef_
    }).sort_values('coefficient', ascending=False)

    print("\n📊 Reduced Model Coefficients:")
    print(coef_reduced.to_string(index=False))

else:
    print("\n⚠️ No features with p-value < 0.05. All features are insignificant.")

print("\n✅ Model diagnostics completed")

# ============================================================
# 7. FORECAST VISUALIZATION
# ============================================================

print("\n" + "="*60)
print("7. FORECAST VISUALIZATION")
print("="*60)

# 7.1. Historical vs predicted plot
plt.figure(figsize=(14, 7))

# Historical data (all)
plt.plot(df.index, df['DEPOS'], label='Actual Data', color='#1f77b4', linewidth=2.5)

# Best model predictions on test set
best_pred = results[best_model_name]['predictions']
test_dates = X.index[train_size:]

plt.plot(test_dates, best_pred, label=f'Forecast ({best_model_name})',
         color='#ff7f0e', linestyle='--', linewidth=2.5)

# Confidence interval (±2 RMSE)
rmse_best = results[best_model_name]['RMSE']
plt.fill_between(test_dates,
                 best_pred - 2*rmse_best,
                 best_pred + 2*rmse_best,
                 alpha=0.25, color='#ff7f0e', label='95% Confidence Interval')

plt.title('Forecasting Household Deposits in Russia\n' +
          f'Best Model: {best_model_name} (R² = {results_df.loc[best_model_name, "R²"]:.4f})',
          fontsize=14)
plt.xlabel('Date')
plt.ylabel('Deposits Volume, billion RUB')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('forecast_plot.png', dpi=300, bbox_inches='tight')
plt.show()

# 7.2. Actual vs Predicted (Scatter plot)
plt.figure(figsize=(8, 8))

all_y = pd.concat([y_train, y_test])
all_pred = pd.concat([
    pd.Series(results[best_model_name]['model'].predict(X_train_scaled if best_model_name == 'Ridge Regression' else X_train),
              index=y_train.index),
    pd.Series(best_pred, index=y_test.index)
])

plt.scatter(all_y, all_pred, alpha=0.6)
plt.plot([all_y.min(), all_y.max()], [all_y.min(), all_y.max()],
         'r--', linewidth=2, label='Ideal Line')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'Actual vs Predicted ({best_model_name})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()

# 7.3. Feature importance (Random Forest only)
if 'Random Forest' in results:
    rf_model = results['Random Forest']['model']
    importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)

    plt.figure(figsize=(10, 8))
    plt.barh(importance['feature'], importance['importance'], color='steelblue')
    plt.xlabel('Importance')
    plt.title('Feature Importance (Random Forest)')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("\n📊 TOP-5 MOST IMPORTANT FEATURES:")
    print(importance.head(5))

# ============================================================
# 8. EXPORT RESULTS
# ============================================================

print("\n" + "="*60)
print("8. EXPORT RESULTS")
print("="*60)

# Create results DataFrame
forecast_df = pd.DataFrame({
    'Date': test_dates,
    'Actual': y_test,
    f'Predicted_{best_model_name}': best_pred,
    'Lower_95%_CI': best_pred - 2*rmse_best,
    'Upper_95%_CI': best_pred + 2*rmse_best
})

# Save to CSV
forecast_df.to_csv('forecast_results.csv', index=False)
print("✅ Forecast results saved to 'forecast_results.csv'")

# Save metrics
results_df.to_csv('model_metrics.csv')
print("✅ Model metrics saved to 'model_metrics.csv'")

# ============================================================
# 9. EXECUTIVE SUMMARY (UPDATED)
# ============================================================

print("\n" + "="*60)
print("9. EXECUTIVE SUMMARY")
print("="*60)

print(f"""
📌 KEY FINDINGS:

1. BEST PERFORMING MODEL: {best_model_name}
   - R² = {results[best_model_name]['R²']:.4f}
   - MAE = {results[best_model_name]['MAE']:.2f} billion RUB
   - RMSE = {results[best_model_name]['RMSE']:.2f} billion RUB

2. RIDGE REGRESSION COEFFICIENTS (TOP DRIVERS):
""")

# Get coefficients from best model (if Ridge)
if 'Ridge' in best_model_name:
    best_coef_df = pd.DataFrame({
        'feature': feature_names,
        'coefficient': results[best_model_name]['model'].coef_
    }).sort_values('coefficient', ascending=False)

    print("   🔺 TOP DRIVERS (increase deposits):")
    for _, row in best_coef_df.head(3).iterrows():
        print(f"      - {row['feature']}: {row['coefficient']:.4f}")

    print("   🔻 TOP DRAGGERS (decrease deposits):")
    negative_coefs = best_coef_df[best_coef_df['coefficient'] < 0].sort_values('coefficient')
    for _, row in negative_coefs.head(3).iterrows():
        print(f"      - {row['feature']}: {row['coefficient']:.4f}")

print(f"""
3. STATISTICAL DIAGNOSTICS:
   - Durbin-Watson Statistic: {dw_stat:.3f} {'✅' if 1.5 < dw_stat < 2.5 else '⚠️'}
   - Conclusion: {'No' if 1.5 < dw_stat < 2.5 else 'Some'} autocorrelation detected

4. KEY TAKEAWAYS:
   - Multicollinearity (VIF > 100) was successfully handled by Ridge regularization
   - Lag features (1, 3, 6, 12 months) improve forecast accuracy
   - Linear Regression with p-value filtering shows which features truly matter
   - Random Forest underperforms on this dataset (small sample size)

5. NEXT STEPS:
   - Add seasonal components (month of year)
   - Test SARIMA or Prophet models
   - Incorporate macroeconomic forecasts for 2026-2027
""")

print("\n✅ PROJECT COMPLETED SUCCESSFULLY!")